In [3]:
import re
import numpy as np
import pandas as pd

from sentence_transformers import (
    SentenceTransformer,
    util,
)

# Load lightweight deployable model
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Model Loaded Successfully")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model Loaded Successfully


In [74]:
messages = [

    "A/c debited by Rs 240 for UPI payment to Swiggy",

    "Salary credited by Employer Corp Rs 85000",

    "Paid Rs 1200 at Dominos",

    "Netflix subscription renewed for Rs 499",

    "UPI transfer of Rs 650 to Rahul Sharma",

    "Amazon order payment of Rs 2200",

    "Uber ride payment of Rs 340",

    "Spotify subscription paid Rs 119",

    "Received Rs 5000 from Amit Patil",

    "Electricity bill payment of Rs 1800",
]

In [75]:
def clean_message(message):

    lower = message.lower()

    blocked_keywords = [

        "otp",
        "cashback",
        "offer",
        "reward",
        "bonus",
        "loan",
    ]

    for keyword in blocked_keywords:

        if keyword in lower:
            return None

    return message.strip()

In [76]:
def extract_amount(message):

    match = re.search(
        r"(?:rs\.?|inr)\s*(\d+(?:\.\d+)?)",
        message,
        re.IGNORECASE,
    )

    if match:
        return float(
            match.group(1)
        )

    return 0

In [77]:
def detect_transaction_type(message):

    lower = message.lower()

    if (
        "credited" in lower
        or "received" in lower
    ):
        return "credit"

    return "debit"

In [78]:
INTENT_PATTERNS = {

    "salary": [
        "salary",
        "credited by",
    ],

    "subscription": [
        "subscription",
        "renewed",
    ],

    "transfer": [
        "upi",
        "transfer",
        "received",
        "sent",
    ],

    "shopping": [
        "order",
        "purchase",
        "amazon",
        "flipkart",
    ],

    "travel": [
        "uber",
        "ola",
        "ride",
        "metro",
    ],

    "bill": [
        "electricity",
        "water bill",
        "gas bill",
        "recharge",
    ],
}


def detect_intent(message):

    lower = message.lower()

    for intent, keywords in INTENT_PATTERNS.items():

        for keyword in keywords:

            if keyword in lower:
                return intent

    return "generic"

In [79]:
GRAMMAR_PATTERNS = [

    r"to\s(.+?)(?:\son|$)",

    r"from\s(.+?)(?:\son|$)",

    r"credited by\s(.+?)(?:\srs|$)",

    r"at\s(.+?)(?:\son|$)",

    r"([A-Za-z]+)\ssubscription",

    r"([A-Za-z]+)\sride",

    r"([A-Za-z]+)\sorder",
]


def extract_merchant(message):

    for regex in GRAMMAR_PATTERNS:

        match = re.search(
            regex,
            message,
            re.IGNORECASE,
        )

        if match:

            merchant = (
                match.group(1)
                .strip()
            )

            merchant = re.sub(
                r"\srs.*",
                "",
                merchant,
                flags=re.IGNORECASE,
            )

            merchant = re.sub(
                r"[^A-Za-z0-9 &.-]",
                "",
                merchant,
            )

            return merchant.strip()

    return "Unknown"

In [80]:
CATEGORY_CONTEXTS = {

    "Food":
    "food restaurant dining cafe swiggy zomato dominos eating delivery",

    "Shopping":
    "shopping ecommerce amazon flipkart myntra products orders buying",

    "Travel":
    "travel cab ride transport uber ola metro fuel",

    "Entertainment":
    "movies music streaming netflix spotify subscription prime",

    "Income":
    "salary credited income received payment received",

    "Transfer":
    "upi transfer bank transfer sent received payment",

    "Bills":
    "electricity water gas utility recharge broadband",
}

In [81]:
merchant_memory = {

    "Swiggy": "Food",

    "Zomato": "Food",

    "Dominos": "Food",

    "Uber": "Travel",

    "Ola": "Travel",

    "Netflix": "Entertainment",

    "Spotify": "Entertainment",

    "Amazon": "Shopping",

    "Flipkart": "Shopping",
}

In [82]:
def semantic_categorize(
    message,
    merchant,
    intent,
):

    # -------------------------
    # MEMORY LAYER
    # -------------------------

    if merchant in merchant_memory:

        return {

            "category":
            merchant_memory[merchant],

            "confidence": 0.95,

            "source": "memory",
        }

    # -------------------------
    # SEMANTIC AI LAYER
    # -------------------------

    scores = {}

    message_embedding = model.encode(
        message,
        convert_to_tensor=True,
    )

    merchant_embedding = model.encode(
        merchant,
        convert_to_tensor=True,
    )

    for category, context in CATEGORY_CONTEXTS.items():

        category_embedding = model.encode(
            context,
            convert_to_tensor=True,
        )

        # Message similarity
        msg_score = util.cos_sim(
            message_embedding,
            category_embedding,
        )[0][0]

        # Merchant similarity
        merchant_score = util.cos_sim(
            merchant_embedding,
            category_embedding,
        )[0][0]

        # Weighted fusion
        final_score = (
            0.5 * float(msg_score)
            + 0.3 * float(merchant_score)
        )

        # -------------------------
        # INTENT PRIORS
        # -------------------------

        if (
            intent == "salary"
            and category == "Income"
        ):
            final_score += 0.4

        if (
            intent == "subscription"
            and category == "Entertainment"
        ):
            final_score += 0.3

        if (
            intent == "shopping"
            and category == "Shopping"
        ):
            final_score += 0.3

        if (
            intent == "travel"
            and category == "Travel"
        ):
            final_score += 0.3

        if (
            intent == "bill"
            and category == "Bills"
        ):
            final_score += 0.3

        if (
            intent == "transfer"
            and category == "Transfer"
        ):
            final_score += 0.2

        scores[category] = final_score

    best_category = max(
        scores,
        key=scores.get,
    )

    confidence = scores[
        best_category
    ]

    # -------------------------
    # SELF-LEARNING MEMORY
    # -------------------------

    if confidence > 0.55:

        merchant_memory[
            merchant
        ] = best_category

    return {

        "category": best_category,

        "confidence": round(
            confidence,
            3,
        ),

        "source": "semantic_ai",
    }

In [83]:
def parse_transaction(message):

    cleaned = clean_message(
        message
    )

    if not cleaned:
        return None

    merchant = extract_merchant(
        cleaned
    )

    intent = detect_intent(
        cleaned
    )

    semantic_result = semantic_categorize(
        cleaned,
        merchant,
        intent,
    )

    return {

        "message": cleaned,

        "merchant": merchant,

        "intent": intent,

        "amount": extract_amount(
            cleaned
        ),

        "type": detect_transaction_type(
            cleaned
        ),

        "category":
        semantic_result["category"],

        "confidence":
        semantic_result["confidence"],

        "source":
        semantic_result["source"],
    }

In [84]:
results = []

for msg in messages:

    parsed = parse_transaction(msg)

    if parsed:
        results.append(parsed)

df = pd.DataFrame(results)

df

,message,merchant,intent,amount,type,category,confidence,source
0,A/c debited by Rs 240 for UPI payment to Swiggy,Swiggy,transfer,240.0,debit,Food,0.950,memory
1,Salary credited by Employer Corp Rs 85000,Employer Corp,salary,85000.0,credit,Income,0.747,semantic_ai
2,Paid Rs 1200 at Dominos,Dominos,generic,1200.0,debit,Food,0.950,memory
3,Netflix subscription renewed for Rs 499,Netflix,subscription,499.0,debit,Entertainment,0.950,memory
4,UPI transfer of Rs 650 to Rahul Sharma,Rahul Sharma,transfer,650.0,debit,Transfer,0.441,semantic_ai
5,Amazon order payment of Rs 2200,Amazon,shopping,2200.0,debit,Shopping,0.950,memory
6,Uber ride payment of Rs 340,Uber,travel,340.0,debit,Travel,0.950,memory
7,Spotify subscription paid Rs 119,Spotify,subscription,119.0,debit,Entertainment,0.950,memory
8,Received Rs 5000 from Amit Patil,Amit Patil,transfer,5000.0,credit,Transfer,0.385,semantic_ai
9,Electricity bill payment of Rs 1800,Unknown,bill,1800.0,debit,Bills,0.502,semantic_ai


In [85]:
merchant_memory

{'Swiggy': 'Food',
 'Zomato': 'Food',
 'Dominos': 'Food',
 'Uber': 'Travel',
 'Ola': 'Travel',
 'Netflix': 'Entertainment',
 'Spotify': 'Entertainment',
 'Amazon': 'Shopping',
 'Flipkart': 'Shopping',
 'Employer Corp': 'Income'}